# Glowtrace training v8 on Kaggle (free GPU): skin texture with both labelling rounds

About 2 to 3 hours of GPU. Retrains the skin surface model (texture, shine, breakouts) on close-up
skin patches from every licensed skin source: 12 Roboflow sets (including real selfies with marked
pores and roughness), Google SCIN, FairFace, Unsplash Lite faces and your team photos.

**New in v8:** Debbie's second labelling round (real face skin, every skin tone; 426 answers
across both rounds, in the updated `glowtrace-labels` dataset).

**From v7:** v6 learnt to ignore blur, because camera blur was added to each view separately and
unlabelled. v7 shares lens and phone processing between both views, anchors a blurred copy of each
patch to its own measured detail, and judges blur against what it does to measured detail. In a
short test this took texture from r 0.68 to 0.95 and blur from 57% to 84% of the ideal.
Also: the fine-detail input channel (v6) and Debbie's "which is smoother" answers (`glowtrace-labels`).

**Each run**
1. **Create → New Notebook → File → Import Notebook**, and choose this file.
2. Right-hand panel: **Accelerator → GPU T4 x2**, **Internet on**.
3. **Input → Add Input → Your Datasets**: add `glowtrace-team` (and `glowtrace-labels` once Debbie's answers are in).
4. **Add-ons → Secrets**: tick `ROBOFLOW_API_KEY` for this notebook.
5. **Save Version → Save & Run All (Commit) → Save**. You can close the tab.
6. When it's finished: open the version → **Output** → download `glowtrace_results_v8.zip` and send it to Claude.

**Built not to waste a run:** cell 1 checks GPU, internet, key and inputs within a minute and says exactly what to fix.
Cell 5 is a 2-minute practice run of the whole pipeline. Training saves as it goes, and the results zip is always made,
with the full log, even if something fails.

In [ ]:
# 1. Pre-flight: checks everything before any long step; stops with a plain message if something is missing.
import os, glob, shutil, json, subprocess, urllib.request, urllib.error, torch
ROOT = os.environ.get('GLOWTRACE_ROOT', '/kaggle')
problems, notes = [], []
if not torch.cuda.is_available() and not os.environ.get('GLOWTRACE_ALLOW_CPU'):
    problems.append('No GPU: in the right-hand panel set Accelerator to GPU T4 x2.')
try:
    urllib.request.urlopen('https://storage.googleapis.com/dx-scin-public-data/dataset/scin_cases.csv', timeout=30).read(100)
except Exception:
    problems.append('No internet: in the right-hand panel turn Internet on (needs a verified phone number).')
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['ROBOFLOW_API_KEY'] = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
except Exception:
    problems.append('Roboflow key missing: Add-ons > Secrets > add ROBOFLOW_API_KEY and tick it for this notebook.')
if os.environ.get('ROBOFLOW_API_KEY') and not os.environ.get('GLOWTRACE_SKIP_ROBOFLOW'):
    try:
        urllib.request.urlopen('https://api.roboflow.com/?api_key=' + os.environ['ROBOFLOW_API_KEY'], timeout=30).read(100)
    except urllib.error.HTTPError as error:
        if error.code in (401, 403):
            problems.append('Roboflow rejected the key: copy a fresh one from roboflow.com > Settings > API Keys into Add-ons > Secrets > ROBOFLOW_API_KEY.')
    except Exception:
        notes.append('Could not reach Roboflow to check the key; carrying on.')
if not glob.glob(f'{ROOT}/input/**/glowtrace_team', recursive=True):
    notes.append('Team photos not attached (Input > Your Datasets > glowtrace-team). Training continues without them.')
label_npz = glob.glob(f'{ROOT}/input/**/label_patches*.npz', recursive=True)
LABELS = os.path.dirname(label_npz[0]) if label_npz else None
if LABELS:
    import numpy as np
    # Every labelling round has its own patch file; answers count only for a round that's attached.
    set_ids = {str(np.load(path)['set_id']) for path in glob.glob(f'{LABELS}/**/label_patches*.npz', recursive=True)}
    per_set = {}
    for path in glob.glob(f'{LABELS}/**/*.json', recursive=True):
        try:
            data = json.load(open(path))
        except Exception:
            continue
        if isinstance(data, dict) and data.get('setId') in set_ids:
            per_set[data['setId']] = per_set.get(data['setId'], 0) + sum(a.get('choice') in ('a', 'b', 'same') for a in data.get('answers', []))
    answers = sum(per_set.values())
    if answers < 50:
        notes.append(f"Debbie's labels attached but only {answers} usable answers: training without them.")
        LABELS = None
    else:
        notes.append(f"Debbie's labels: {answers} answers across {len(per_set)} rounds " + json.dumps(per_set) + ".")
else:
    notes.append("Debbie's labels not attached: training without them (fine; add glowtrace-labels when ready).")
for n in notes: print('note:', n)
if problems:
    raise SystemExit('Fix these, then Save Version again:\n- ' + '\n- '.join(problems))
print('Pre-flight passed.')

In [ ]:
# 2. Tools and folders.
!pip install -q onnx roboflow mediapipe pandas pyarrow
WORK = os.environ.get('GLOWTRACE_WORKDIR', f'{ROOT}/working')
os.chdir(WORK)
OUT = f'{WORK}/results'
os.makedirs(OUT, exist_ok=True)
os.environ.setdefault('GLOWTRACE_WORK', f'{WORK}/cache')
LOG = open(f'{OUT}/run_log.txt', 'a')
def log(message):
    print(message, flush=True); LOG.write(message + '\n'); LOG.flush()
log('setup done')

In [ ]:
%%writefile multitask.py
"""Glowtrace vision: one physics-supervised model per area (skin, hair, nails), every indicator.

Nobody labels "how much redness, shine, ridging or yellowing" a photo has, so the labels come from
physics and optics. Real base photos get known amounts of each effect added, per pixel:

  skin   redness   haemoglobin absorption (Beer-Lambert, OD direction R:G:B 0.04 : 1 : 0.85)
         pigment   melanin absorption in patches (OD 0.45 : 0.75 : 1), freckle- to patch-sized
         shine     surface (specular) reflection: adds the light's own colour (dichromatic model)
         texture   fine bumps: high-frequency shading of brightness, like pores and roughness
  nails  yellowing stain absorbing blue most (OD 0.05 : 0.25 : 1)
         spots     small white spots blended in (leukonychia-like)
         ridges    lengthwise shading ripple along the nail
  hair   shine     elongated highlight bands
         texture   fine high-frequency roughness

Each training example is two views of one base photo, each under its own random light colour,
exposure, blur and sensor noise; view B also gets a random mix of the effects. A small U-Net
with one output map per indicator must predict, at every pixel, how much of each effect B has
over A. So it learns to separate the effects from each other and from light and camera. For skin
a fairness term keeps the average output equal across Monk skin-tone groups.

Skin base photos: Google SCIN (SCIN Data Use License, no re-identification). Hair and nail base
photos: a folder of licensed images (--images DIR), see research/vision/README.md.

Usage:
  python3 multitask.py skin [--quick]
  python3 multitask.py nails --images DIR [--quick]
  python3 multitask.py hair --images DIR [--quick]
Writes <area>_model.onnx, <area>_weights.json and <area>_eval.json in the current folder.
"""
import csv
import glob
import io
import json
import os
import sys
import urllib.request
import zlib
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

AREA = sys.argv[1] if len(sys.argv) > 1 else "skin"
QUICK = "--quick" in sys.argv
IMAGES_DIR = sys.argv[sys.argv.index("--images") + 1] if "--images" in sys.argv else None
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Fine detail (nail spots and ridges, hair shine and breakage, scalp flakes) needs more pixels
# than 128. Pass --size 256 for nails and hair; the app reads the size from the weights file.
SIZE = int(sys.argv[sys.argv.index("--size") + 1]) if "--size" in sys.argv else 128
# v5 skin surface: train on close-up skin patches (pores a pixel or two across) instead of whole
# photos shrunk to SIZE, with realistic pores, side-lit bumps, beauty-filter smoothing and phone
# camera effects. The texture label is measured, not assumed: the change in fine-scale contrast.
PATCHES = "--patches" in sys.argv
PATCHES_PER_PHOTO = 3
# v6: a fifth input channel, fine detail. Skin texture is a 1-2% brightness change on top of a
# far larger colour signal; in the 4-channel input the network could not even reproduce the
# texture measure (r 0.27 on labelled patches). The detail channel (luma optical density minus its
# 7x7 reflect-padded box mean, times DETAIL_GAIN) lifts that to r 0.95. The app builds the same
# channel when the weights file says "detail" (src/lib/cv/area-model.ts).
DETAIL_GAIN = 30.0 if PATCHES else 0.0
IN_CHANNELS = 5 if DETAIL_GAIN else 4
# v6: a person's "which patch is smoother" answers (make_label_page.py). --labels DIR holds
# label_patches.npz and one or more glowtrace_labels_*.json files from the labelling page.
LABELS_DIR = sys.argv[sys.argv.index("--labels") + 1] if "--labels" in sys.argv else None
EPOCHS = int(os.environ.get("GLOWTRACE_EPOCHS", 2 if QUICK else 80))
STEPS = int(os.environ.get("GLOWTRACE_STEPS", 4 if QUICK else 50))
# Each head's loss divided by its own target energy, so sparse effects (spots, flakes, breakage,
# shine streaks) count as much as effects that cover the whole area. Off only for comparison.
BALANCE = os.environ.get("GLOWTRACE_BALANCE", "1") == "1"
WORK = os.environ.get("GLOWTRACE_WORK", "glowtrace_work")

ALL_HEADS = {
    "skin": ["redness", "pigment", "shine", "texture", "breakouts", "dark_circles"],
    "nails": ["yellowing", "spots", "ridges", "cuticle"],
    "hair": ["shine", "texture", "flakes"],
}[AREA]
# Smaller models learn better: --heads picks which effects this model reads (the others are still
# added to the photos, unlabelled, so the model learns to ignore them). --name names the files.
HEADS = sys.argv[sys.argv.index("--heads") + 1].split(",") if "--heads" in sys.argv else ALL_HEADS
NAME = sys.argv[sys.argv.index("--name") + 1] if "--name" in sys.argv else AREA
OWN = [ALL_HEADS.index(h) for h in HEADS]
assert set(HEADS) <= set(ALL_HEADS), f"unknown heads {set(HEADS) - set(ALL_HEADS)}"
# Hair: effects go on hair pixels only, found by MediaPipe's hair segmenter (the app's own guide).
HAIR_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/image_segmenter/hair_segmenter/float32/latest/hair_segmenter.tflite"

HAEMOGLOBIN = torch.tensor([0.04, 1.0, 0.85])
# Dark circles: deoxygenated blood under thin skin plus a little melanin. An approximation of the
# relative absorption in r, g, b; the head learns the direction of change, not a clinical amount.
DARK_CIRCLE = torch.tensor([0.55, 0.75, 0.6])
MELANIN = torch.tensor([0.45, 0.75, 1.0])
STAIN = torch.tensor([0.05, 0.25, 1.0])

# ---------------------------------------------------------------- base photos

BUCKET = "https://storage.googleapis.com/dx-scin-public-data/"
GROUPS = {0: [1, 2], 1: [3, 4], 2: [5, 6], 3: [7, 8, 9, 10]}


def fetch(url):
    with urllib.request.urlopen(url, timeout=60) as response:
        return response.read()


def to_examples(image, key=""):
    """Whole photo as one canvas, or (--patches) up to three skin-rich SIZE x SIZE crops taken at a
    random scale (short side 256-640 px), so pores and bumps keep a realistic pixel size."""
    if not PATCHES:
        return [to_canvas(image)]
    rng = np.random.default_rng(zlib.crc32(key.encode()))
    image = image.convert("RGB")
    short = rng.uniform(256, 640)
    scale = short / min(image.size)
    w, h = max(SIZE, round(image.width * scale)), max(SIZE, round(image.height * scale))
    array = np.asarray(image.resize((w, h), Image.BILINEAR))
    skin = skin_rule(array)
    found = []
    for _ in range(16):
        y, x = rng.integers(0, h - SIZE + 1), rng.integers(0, w - SIZE + 1)
        found.append((skin[y : y + SIZE, x : x + SIZE].mean(), y, x))
    found.sort(reverse=True)
    out = []
    for share, y, x in found[:PATCHES_PER_PHOTO]:
        if share < 0.6:
            break
        out.append((array[y : y + SIZE, x : x + SIZE].copy(), np.ones((SIZE, SIZE), np.uint8)))
    return out


def skin_rule(array):
    """The app's skin-colour rule on an HxWx3 uint8 array (numpy twin of area_mask)."""
    L, a, b = lab(torch.tensor(array, dtype=torch.float32).permute(2, 0, 1)[None])
    hue = torch.rad2deg(torch.atan2(b, a))
    return ((L > 20) & (L < 95) & (a > 2) & (hue > 10) & (hue < 80))[0].numpy()


def to_canvas(image):
    image.thumbnail((SIZE, SIZE), Image.BILINEAR)
    canvas = np.zeros((SIZE, SIZE, 3), np.uint8)
    valid = np.zeros((SIZE, SIZE), np.uint8)
    array = np.asarray(image.convert("RGB"))
    canvas[: array.shape[0], : array.shape[1]] = array
    valid[: array.shape[0], : array.shape[1]] = 1
    return canvas, valid


def load_scin(per_group):
    os.makedirs(WORK, exist_ok=True)
    cache = os.path.join(WORK, f"scin_{per_group}_{SIZE}{'_patches' if PATCHES else ''}.npz")
    if os.path.exists(cache):
        # A cache left half-written by an interrupted run is ignored and rebuilt, never trusted.
        try:
            d = np.load(cache)
            return d["image"], d["valid"], d["group"]
        except Exception:
            print(f"Rebuilding unreadable cache {cache}")
            os.remove(cache)
    cases = {r["case_id"]: r for r in csv.DictReader(io.StringIO(fetch(BUCKET + "dataset/scin_cases.csv").decode()))}
    labels = list(csv.DictReader(io.StringIO(fetch(BUCKET + "dataset/scin_labels.csv").decode())))
    picks = []
    for group, tones in GROUPS.items():
        rows = []
        for label in labels:
            try:
                tone = int(label["monk_skin_tone_label_us"])
            except ValueError:
                continue
            case = cases.get(label["case_id"])
            if tone in tones and case and case["image_1_shot_type"] == "CLOSE_UP" and case["image_1_path"]:
                rows.append((label["case_id"], case["image_1_path"], group))
        rows.sort()
        picks += rows[:per_group]

    def one(item):
        try:
            examples = to_examples(Image.open(io.BytesIO(fetch(BUCKET + item[1]))), item[1])
            return [(canvas, valid, item[2]) for canvas, valid in examples]
        except Exception:
            return None

    with ThreadPoolExecutor(8) as pool:
        loaded = [x for found in pool.map(one, picks) if found for x in found]
    out = (np.stack([x[0] for x in loaded]), np.stack([x[1] for x in loaded]), np.array([x[2] for x in loaded]))
    np.savez_compressed(cache, image=out[0], valid=out[1], group=out[2])
    return out


def load_folder(folder, limit):
    # A fair share from every dataset (top-level folder), so one large set cannot crowd out others.
    by_set = {}
    for p in sorted(glob.glob(os.path.join(folder, "**", "*.*"), recursive=True)):
        if p.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            by_set.setdefault(os.path.relpath(p, folder).split(os.sep)[0], []).append(p)
    share = max(1, limit // max(1, len(by_set)))
    paths = []
    for top, items in sorted(by_set.items()):
        np.random.default_rng(0).shuffle(items)
        paths += items[:share]
        print(f"  {top}: using {min(share, len(items))} of {len(items)}")
    loaded = []
    for path in paths:
        try:
            loaded += to_examples(Image.open(path), path)
        except Exception:
            continue
    if not loaded:
        raise SystemExit(f"No images found in {folder}")
    images = np.stack([x[0] for x in loaded])
    valid = np.stack([x[1] for x in loaded])
    return images, valid, np.zeros(len(loaded), dtype=int)


# ---------------------------------------------------------------- colour and masks


def lab(rgb):
    c = rgb / 255.0
    lin = torch.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)
    r, g, b = lin[:, 0], lin[:, 1], lin[:, 2]
    x = (r * 0.4124564 + g * 0.3575761 + b * 0.1804375) / 0.95047
    y = r * 0.2126729 + g * 0.7151522 + b * 0.072175
    z = (r * 0.0193339 + g * 0.119192 + b * 0.9503041) / 1.08883
    f = lambda t: torch.where(t > 216 / 24389, torch.pow(t.clamp(min=1e-9), 1 / 3), (841 / 108) * t + 4 / 29)
    fx, fy, fz = f(x), f(y), f(z)
    return 116 * fy - 16, 500 * (fx - fy), 200 * (fy - fz)


def area_mask(rgb, valid):
    """Skin: skin-coloured pixels (the app's rule). Hair: hair pixels (valid == 2, from
    hair_masks) when at least 15% of the photo is hair. Nails: the whole valid photo."""
    if AREA == "hair":
        hair = valid > 1
        enough = (hair.flatten(1).float().mean(1) >= 0.15)[:, None, None]
        return torch.where(enough, hair, valid > 0).float()
    if AREA != "skin":
        return (valid > 0).float()
    L, a, b = lab(rgb)
    hue = torch.rad2deg(torch.atan2(b, a))
    skin = (L > 20) & (L < 95) & (a > 2) & (hue > 10) & (hue < 80) & (valid > 0)
    fallback = (skin.flatten(1).float().mean(1) < 0.1)[:, None, None]
    return torch.where(fallback, valid > 0, skin).float()


def od(rgb):
    return -torch.log((rgb + 0.5) / 255.5)


def from_od(density):
    return 255.5 * torch.exp(-density) - 0.5


def network_input(rgb, mask):
    density = od(rgb)
    parts = [density / 3.0]
    if DETAIL_GAIN:
        luma = density.mean(1, keepdim=True)
        parts.append((luma - box(luma, 7)) * DETAIL_GAIN)
    return torch.cat(parts + [mask[:, None]], 1)


# ---------------------------------------------------------------- effect generators (per-pixel weights)


def rand(shape, gen, lo=0.0, hi=1.0):
    return lo + (hi - lo) * torch.rand(shape, device=DEVICE, generator=gen)


def grid(n):
    yy = torch.arange(SIZE, device=DEVICE).view(1, SIZE, 1).float()
    xx = torch.arange(SIZE, device=DEVICE).view(1, 1, SIZE).float()
    return yy, xx


def blobs(n, gen, count, small, large):
    """Sum of `count` soft round blobs per image, each 0-1."""
    yy, xx = grid(n)
    out = torch.zeros(n, SIZE, SIZE, device=DEVICE)
    for _ in range(count):
        cy, cx = rand((n, 1, 1), gen, 0, SIZE), rand((n, 1, 1), gen, 0, SIZE)
        sigma = rand((n, 1, 1), gen, small, large)
        out = torch.maximum(out, torch.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * sigma**2)))
    return out


def streaks(n, gen, count, length, width):
    """Elongated highlight bands at a random angle per image (hair shine)."""
    yy, xx = grid(n)
    angle = rand((n, 1, 1), gen, 0, np.pi)
    u = xx * torch.cos(angle) + yy * torch.sin(angle)
    v = -xx * torch.sin(angle) + yy * torch.cos(angle)
    out = torch.zeros(n, SIZE, SIZE, device=DEVICE)
    for _ in range(count):
        cu, cv = rand((n, 1, 1), gen, -SIZE, 2 * SIZE), rand((n, 1, 1), gen, -SIZE, 2 * SIZE)
        band = torch.exp(-((u - cu) ** 2) / (2 * length**2) - ((v - cv) ** 2) / (2 * width**2))
        out = torch.maximum(out, band)
    return out


def fine_noise(n, gen):
    """Band-limited high-frequency field in -1..1 (pores, bumps, fibres)."""
    noise = torch.randn(n, 1, SIZE, SIZE, device=DEVICE, generator=gen)
    smooth = F.avg_pool2d(F.pad(noise, (1, 1, 1, 1), mode="reflect"), 3, 1)
    coarse = F.avg_pool2d(F.pad(noise, (3, 3, 3, 3), mode="reflect"), 7, 1)
    field = (smooth - coarse)[:, 0]
    return field / field.flatten(1).std(1).clamp(min=1e-6)[:, None, None]


def box(x, k):
    p = k // 2
    return F.avg_pool2d(F.pad(x, (p, p, p, p), mode="reflect"), k, 1)


def hp_contrast(img, mask):
    """Fine-scale relative contrast of brightness inside the mask: the std of (luma minus its 7 px
    local mean) over that mean. This is the texture unit for --patches training."""
    lum = img.mean(1, keepdim=True) + 1.0
    base = box(lum, 7)
    rel = ((lum - base) / base)[:, 0]
    weight = mask.flatten(1).sum(1).clamp(min=1)
    mean = (rel * mask).flatten(1).sum(1) / weight
    var = (((rel - mean[:, None, None]) ** 2) * mask).flatten(1).sum(1) / weight
    return var.sqrt()


def skin_texture(img, mask, gen, amount):
    """Real-looking surface change on skin patches. Returns the photo and the texture map: the
    measured change in hp_contrast, so pores, bumps and smoothing all share one honest unit."""
    n = img.shape[0]
    before = hp_contrast(img, mask)
    m = mask[:, None]
    # Pores: sparse soft dark dots, 1-3 px, denser on some patches than others.
    field = box(torch.randn(n, 1, SIZE, SIZE, device=DEVICE, generator=gen), 3)
    q = rand((n,), gen, 0.9, 0.985)
    ordered = field.flatten(1).sort(1).values
    thresh = ordered.gather(1, (q * (SIZE * SIZE - 1)).long()[:, None]).view(n, 1, 1, 1)
    pores = box((field > thresh).float(), 3)
    img = img * (1 - amount(0.0, 0.25)[:, None] * pores * m)
    # Bumps lit from one side: a 3-11 px height field shaded along a random light direction.
    height = box(torch.randn(n, 1, SIZE, SIZE, device=DEVICE, generator=gen), 3)
    height = height - box(height, 11)
    gy = F.pad(height[:, :, 1:] - height[:, :, :-1], (0, 0, 0, 1))
    gx = F.pad(height[:, :, :, 1:] - height[:, :, :, :-1], (0, 1, 0, 0))
    angle = rand((n, 1, 1, 1), gen, 0, 2 * np.pi)
    shade = gx * torch.cos(angle) + gy * torch.sin(angle)
    shade = shade / shade.flatten(1).std(1).clamp(min=1e-6).view(n, 1, 1, 1)
    img = img * (1 + amount(0.0, 0.04)[:, None] * shade * m)
    # Fine roughness (the v4 effect, now one ingredient of several).
    img = img * (1 + amount(0.0, 0.025)[:, None] * fine_noise(n, gen)[:, None] * m)
    # Beauty filter or soft focus on some photos: smoothing must read as smoother skin.
    smooth = (torch.rand(n, device=DEVICE, generator=gen) < 0.3).float().view(n, 1, 1, 1)
    strength = rand((n, 1, 1, 1), gen, 0.3, 1.0) * smooth * m
    blurred = torch.where(rand((n, 1, 1, 1), gen) < 0.5, box(img, 3), box(img, 5))
    img = img * (1 - strength) + blurred * strength
    change = hp_contrast(img.clamp(0, 255), mask) - before
    return img, change[:, None, None] * mask


def optics(img, gen):
    """Lens and phone processing (--patches only): blur, resampling softness and sharpening, each
    at random strength. Applied to the base photo BEFORE the effects, so both views share it and the
    measured texture label already includes it. v6 applied these to each view separately and
    unlabelled, which taught the texture head to ignore blur (real-photo blur check -0.02)."""
    n = img.shape[0]
    pick = lambda p: torch.rand(n, device=DEVICE, generator=gen) < p
    blur = pick(0.4)
    if blur.any():
        img[blur] = box(img[blur], 3)
    soft = pick(0.3)
    if soft.any():
        small = F.interpolate(img[soft], scale_factor=0.6, mode="bilinear", align_corners=False)
        img[soft] = F.interpolate(small, size=img.shape[-2:], mode="bilinear", align_corners=False)
    sharp = pick(0.4)
    if sharp.any():
        amount_ = rand((int(sharp.sum()), 1, 1, 1), gen, 0.2, 1.0)
        img[sharp] = img[sharp] + amount_ * (img[sharp] - box(img[sharp], 3))
    return img.clamp(0, 255)


def jpeg(img, qualities):
    """JPEG at a given quality per photo (0 = none); both views use the same qualities."""
    try:
        from torchvision.io import decode_jpeg, encode_jpeg
    except ImportError:
        return img
    for i, quality in enumerate(qualities):
        if quality:
            data = encode_jpeg(img[i].clamp(0, 255).round().to(torch.uint8).cpu(), quality=quality)
            img[i] = decode_jpeg(data).float().to(DEVICE)
    return img


def ripple(n, gen):
    """Lengthwise ridges: a sinusoid across the nail at a random angle and spacing."""
    yy, xx = grid(n)
    angle = rand((n, 1, 1), gen, -0.3, 0.3) + np.pi / 2
    period = rand((n, 1, 1), gen, 3, 8)
    u = xx * torch.cos(angle) + yy * torch.sin(angle)
    return torch.sin(2 * np.pi * u / period)


def add_effects(rgb, mask, gen):
    """Adds a random mix of this area's effects. Returns the new photo and the per-head maps."""
    n = rgb.shape[0]
    maps = {}
    density = od(rgb)
    light = 255 * torch.ones(n, 3, 1, 1, device=DEVICE)  # specular adds the light's colour
    img_od = density.clone()

    def amount(lo, hi, p_zero=0.3):
        a = rand((n, 1, 1), gen, lo, hi)
        a[torch.rand(n, device=DEVICE, generator=gen) < p_zero] = 0
        return a

    if AREA == "skin":
        hb = amount(-0.08, 0.25) * torch.where(rand((n, 1, 1), gen) < 0.5, torch.ones(n, SIZE, SIZE, device=DEVICE), blobs(n, gen, 1, 15, 60)) * mask
        mel = amount(0.0, 0.35) * blobs(n, gen, 6, 2, 12) * mask
        img_od = img_od + hb[:, None] * HAEMOGLOBIN.to(DEVICE)[None, :, None, None] + mel[:, None] * MELANIN.to(DEVICE)[None, :, None, None]
        maps["redness"], maps["pigment"] = hb, mel
        img = from_od(img_od)
        if PATCHES:
            img, maps["texture"] = skin_texture(img, mask, gen, amount)
        else:
            rough = amount(0.0, 0.12) * mask
            img = img * (1 + rough[:, None] * fine_noise(n, gen)[:, None])
            maps["texture"] = rough
        spec = amount(0.0, 0.35) * blobs(n, gen, 3, 4, 20) * mask
        img = img + spec[:, None] * light
        maps["shine"] = spec
        # Breakouts: small, sharp-edged red bumps with a tiny highlight on top (unlike the broad
        # redness above). Dark circles: one soft elongated patch of darkening.
        bump = amount(0.0, 0.5) * blobs(n, gen, 5, 1.0, 3.0) * mask
        img = from_od(od(img) + bump[:, None] * HAEMOGLOBIN.to(DEVICE)[None, :, None, None])
        img = img + 0.15 * bump[:, None] * light * (bump[:, None] > 0.6)
        under = amount(0.0, 0.3) * streaks(n, gen, 1, 12, 5) * mask
        img = from_od(od(img) + under[:, None] * DARK_CIRCLE.to(DEVICE)[None, :, None, None])
        maps["breakouts"], maps["dark_circles"] = bump, under
    elif AREA == "nails":
        stain = amount(0.0, 0.5) * mask
        img = from_od(img_od + stain[:, None] * STAIN.to(DEVICE)[None, :, None, None])
        spots = amount(0.0, 0.8) * blobs(n, gen, 4, 1.5, 4) * mask
        img = img * (1 - spots[:, None]) + 245 * spots[:, None]
        ridge = amount(0.0, 0.08) * mask
        img = img * (1 + ridge[:, None] * ripple(n, gen)[:, None])
        # Dry cuticles: a pale, rough band along one edge of the photo's nail area.
        yy, xx = grid(n)
        edge = torch.where(rand((n, 1, 1), gen) < 0.5, yy, SIZE - 1 - yy) / SIZE
        band = torch.exp(-((edge - rand((n, 1, 1), gen, 0.05, 0.3)) ** 2) / (2 * 0.04**2))
        cut = amount(0.0, 0.5) * band * mask
        flake = (fine_noise(n, gen) > 0.8).float()
        img = img * (1 - 0.5 * cut[:, None] * flake[:, None]) + 235 * 0.5 * cut[:, None] * flake[:, None]
        maps["yellowing"], maps["spots"], maps["ridges"], maps["cuticle"] = stain, spots, ridge, cut
    else:
        img = from_od(img_od)
        rough = amount(0.0, 0.15) * mask
        img = img * (1 + rough[:, None] * fine_noise(n, gen)[:, None])
        spec = amount(0.0, 0.4) * streaks(n, gen, 3, 40, 3) * mask
        img = img + spec[:, None] * light
        # Flakes: tiny pale specks, as on the scalp or roots.
        flakes = amount(0.0, 0.7) * blobs(n, gen, 10, 0.6, 1.5) * mask
        img = img * (1 - flakes[:, None]) + 240 * flakes[:, None]
        maps["shine"], maps["texture"], maps["flakes"] = spec, rough, flakes
    # Every effect's map, in ALL_HEADS order; the model is trained on its HEADS columns only.
    return img, torch.stack([maps[h] for h in ALL_HEADS], 1)


def camera(rgb, gen, qualities=None):
    """Random light colour, exposure, blur and sensor noise. With --patches, blur lives in
    optics() (shared by both views) and JPEG uses the shared `qualities`."""
    n = rgb.shape[0]
    gains = torch.exp(0.07 * torch.randn(n, 3, 1, 1, device=DEVICE, generator=gen))
    gains = gains * torch.exp(0.15 * torch.randn(n, 1, 1, 1, device=DEVICE, generator=gen))
    img = rgb * gains
    blur = torch.rand(n, device=DEVICE, generator=gen) < (0.0 if PATCHES else 0.4)
    if blur.any():
        img[blur] = F.avg_pool2d(F.pad(img[blur], (1, 1, 1, 1), mode="replicate"), 3, 1)
    if AREA == "hair":
        # Unlabelled glare (round, not along the strands) on half the photos: the shine head must
        # learn that only highlights running along the hair count.
        glare = blobs(n, gen, 2, 6, 25) * rand((n, 1, 1), gen, 0, 0.35) * (torch.rand(n, 1, 1, device=DEVICE, generator=gen) < 0.5)
        img = img + glare[:, None] * 255
    sigma = rand((n, 1, 1, 1), gen, 0.5, 3.0)
    img = (img + torch.randn(img.shape, device=DEVICE, generator=gen) * sigma).clamp(0, 255)
    if PATCHES and qualities is not None:
        img = jpeg(img, qualities).clamp(0, 255)
    return img.round()


def make_batch(images, valid, gen):
    qualities = None
    if PATCHES:
        images = optics(images.clone(), gen)
        pick = (torch.rand(images.shape[0], device=DEVICE, generator=gen) < 0.5).tolist()
        qualities = [int(rand((1,), gen, 45, 95).item()) if p else 0 for p in pick]
    mask = area_mask(images, valid)
    with_effects, targets = add_effects(images, mask, gen)
    a, b = camera(images, gen, qualities), camera(with_effects, gen, qualities)
    # The 4th channel is the valid-photo mask, exactly what the app feeds.
    seen = (valid > 0).float()
    return network_input(a, seen), network_input(b, seen), targets, mask, images


# ---------------------------------------------------------------- model


class UNet(nn.Module):
    def __init__(self, heads):
        super().__init__()
        conv = lambda i, o, s=1: nn.Sequential(nn.Conv2d(i, o, 3, s, 1), nn.ReLU())
        self.e1, self.e2, self.e3 = conv(IN_CHANNELS, 16), conv(16, 32, 2), conv(32, 48, 2)
        self.mid = conv(48, 48)
        self.d2, self.d1 = conv(80, 32), conv(48, 16)
        self.out = nn.Conv2d(16, heads, 1)

    def forward(self, x):
        s1 = self.e1(x)
        s2 = self.e2(s1)
        s3 = self.mid(self.e3(s2))
        u2 = self.d2(torch.cat([F.interpolate(s3, size=s2.shape[-2:], mode="nearest"), s2], 1))
        u1 = self.d1(torch.cat([F.interpolate(u2, size=s1.shape[-2:], mode="nearest"), s1], 1))
        return self.out(u1)


def masked_mean(values, mask):
    return (values * mask[:, None]).flatten(2).sum(2) / mask.flatten(1).sum(1).clamp(min=1)[:, None]


# ---------------------------------------------------------------- skin-tone groups

TONE_NAMES = ["lighter (ITA > 41)", "light-medium (28-41)", "medium-tan (10-28)", "deep (<= 10)"]


def tone_groups(images, valid):
    """Skin-tone band per photo from ITA (atan((L* - 50) / b*)) of its skin-coloured pixels.
    Used to balance training and check fairness for skin and nails. 1 when too little skin."""
    out = []
    for start in range(0, len(images), 64):
        rgb = images[start : start + 64]
        L, a, b = lab(rgb)
        hue = torch.rad2deg(torch.atan2(b, a))
        skin = (L > 20) & (L < 95) & (a > 2) & (hue > 10) & (hue < 80) & (valid[start : start + 64] > 0)
        for i in range(rgb.shape[0]):
            if skin[i].sum() < 200:
                out.append(1)
                continue
            ita = torch.rad2deg(torch.atan((L[i][skin[i]].median() - 50) / b[i][skin[i]].median().clamp(min=1e-3)))
            out.append(0 if ita > 41 else 1 if ita > 28 else 2 if ita > 10 else 3)
    return torch.tensor(out, device=images.device)


# ---------------------------------------------------------------- train and evaluate


def hair_masks(images_np, valid_np):
    """Marks hair pixels as 2 in the valid mask. Skipped (whole photo) if MediaPipe is missing."""
    try:
        import mediapipe as mp
        from mediapipe.tasks.python import vision, BaseOptions
    except (ImportError, OSError) as error:
        print(f"MediaPipe unavailable ({error}): effects go on the whole photo")
        return valid_np
    path = os.path.join(WORK, "hair_segmenter.tflite")
    os.makedirs(WORK, exist_ok=True)
    if not os.path.exists(path):
        open(path, "wb").write(fetch(HAIR_MODEL_URL))
    segmenter = vision.ImageSegmenter.create_from_options(vision.ImageSegmenterOptions(
        base_options=BaseOptions(model_asset_path=path), output_category_mask=True))
    out = valid_np.copy()
    found = 0
    for i, image in enumerate(images_np):
        result = segmenter.segment(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(image)))
        hair = np.squeeze(result.category_mask.numpy_view()) > 0
        out[i][hair & (valid_np[i] > 0)] = 2
        found += hair.mean() >= 0.15
    segmenter.close()
    print(f"hair found in {found} of {len(images_np)} photos; the rest use the whole photo")
    return out


def load_labels(folder):
    """Labelled pairs from the labelling page. Refuses answers from a different patch set, so a
    label can never be joined to the wrong patch. Returns the patches, train and held-out pairs as
    (a, b, target) with target 1 = a smoother, 0 = b smoother, 0.5 = about the same, and the
    labeller's own consistency on the repeated pairs."""
    # Every labelling round is its own patch file (label_patches*.npz, anywhere in the folder);
    # answers join the round whose set id they carry, with patch numbers offset into one array.
    sets = {}
    images = []
    for npz in sorted(glob.glob(os.path.join(folder, "**", "label_patches*.npz"), recursive=True)):
        data = np.load(npz)
        sid = str(data["set_id"])
        if sid in sets:
            continue
        sets[sid] = sum(len(i) for i in images)
        images.append(data["image"])
    if not sets:
        raise SystemExit(f"--labels {folder}: no label_patches*.npz found")
    set_id = ", ".join(sets)
    pairs = []
    for path in sorted(glob.glob(os.path.join(folder, "**", "*.json"), recursive=True)):
        try:
            answers = json.load(open(path))
        except Exception:
            print(f"labels: skipped unreadable {path}")
            continue
        if not isinstance(answers, dict) or answers.get("format") != "glowtrace-labels-v1":
            continue
        offset = sets.get(str(answers.get("setId")))
        if offset is None:
            print(f"labels: skipped {path}, made for patch set {answers.get('setId')} (have {set_id})")
            continue
        for item in answers.get("answers", []):
            target = {"a": 1.0, "b": 0.0, "same": 0.5}.get(item.get("choice"))
            if target is not None:
                pairs.append((int(item["a"]) + offset, int(item["b"]) + offset, target))
    if not pairs:
        raise SystemExit(f"--labels {folder}: no answers for patch set {set_id}")
    # Consistency: pairs answered twice (once each way round) should agree.
    # Strict: the same answer both times. Not contradicting: never A one time and B the other
    # ("about the same" plus a choice means the two patches were close, not that she disagreed).
    seen, agree, opposite, both = {}, 0, 0, 0
    for a, b, t in pairs:
        key = (min(a, b), max(a, b))
        smoother_a = t if a == key[0] else 1 - t
        if key in seen:
            both += 1
            agree += seen[key] == smoother_a
            opposite += {seen[key], smoother_a} == {0.0, 1.0}
        else:
            seen[key] = smoother_a
    consistency = agree / both if both else None
    if both:
        print(f"labels: repeats {both}, same answer {agree}, opposite {opposite} "
              f"(not contradicting {1 - opposite / both:.2f})")
    # Hold out a fifth of the patch pairs (by pair, both orders together) for evaluation.
    rng = np.random.default_rng(7)
    keys = sorted(seen)
    held = set(map(tuple, np.array(keys)[rng.permutation(len(keys))[: len(keys) // 5]].tolist()))
    train = [p for p in pairs if (min(p[0], p[1]), max(p[0], p[1])) not in held]
    test = [p for p in pairs if (min(p[0], p[1]), max(p[0], p[1])) in held]
    print(f"labels: {len(pairs)} answers, {len(train)} train / {len(test)} held out, self-consistency {consistency}")
    return np.concatenate(images), train, test, consistency


def label_levels(model, patches, ids):
    """Texture reading per labelled patch, exactly as on any photo (whole patch valid)."""
    img = patches[ids]
    valid = torch.ones(img.shape[0], SIZE, SIZE, device=DEVICE)
    k = HEADS.index("texture")
    return masked_mean(model(network_input(img, valid)), area_mask(img, valid))[:, k]


def label_loss(model, patches, pairs, gen, count=32):
    """Bradley-Terry: the smoother patch must read lower texture; "about the same" pairs close."""
    pick = torch.randint(len(pairs), (count,), device=DEVICE, generator=gen).tolist()
    a = torch.tensor([pairs[i][0] for i in pick], device=DEVICE)
    b = torch.tensor([pairs[i][1] for i in pick], device=DEVICE)
    t = torch.tensor([pairs[i][2] for i in pick], device=DEVICE)
    gap = label_levels(model, patches, b) - label_levels(model, patches, a)  # > 0: a smoother
    return F.binary_cross_entropy_with_logits(gap * 4.0, t)


def label_agreement(model, patches, pairs):
    """Share of held-out clear answers (not "about the same") the model agrees with."""
    clear = [p for p in pairs if p[2] != 0.5]
    if not clear:
        return None
    right = 0
    for start in range(0, len(clear), 64):
        chunk = clear[start : start + 64]
        a = torch.tensor([p[0] for p in chunk], device=DEVICE)
        b = torch.tensor([p[1] for p in chunk], device=DEVICE)
        gap = label_levels(model, patches, b) - label_levels(model, patches, a)
        right += sum(int((g > 0) == (p[2] == 1.0)) for g, p in zip(gap.tolist(), chunk))
    return right / len(clear)


def real_photo_checks(model, images, valid):
    """Step 4 of the plan, on untouched held-out real patches (no simulated effects). A blur must
    lower the texture reading by at least 70% of what it does to the measured fine detail
    (hp_contrast) of the same patches, and +/-20% exposure must move the reading by at most a fifth
    of the spread between photos. (v5/v6 asked for blur <= -0.5 of the spread, but the measured
    detail itself only reaches about -0.49 on real skin, so no model could pass.)"""
    k = HEADS.index("texture")
    seen = (valid > 0).float()

    def level(img):
        out = []
        for s in range(0, len(img), 64):
            mask = area_mask(img[s : s + 64], valid[s : s + 64])
            out.append(masked_mean(model(network_input(img[s : s + 64], seen[s : s + 64])), mask)[:, k])
        return torch.cat(out)

    def detail(img):
        return torch.cat([hp_contrast(img[s : s + 64], area_mask(img[s : s + 64], valid[s : s + 64])) for s in range(0, len(img), 64)])

    blurred = box(images, 3)
    base = level(images)
    spread = base.std().clamp(min=1e-6)
    blur = ((level(blurred) - base).mean() / spread).item()
    truth = detail(images)
    ideal = ((detail(blurred) - truth).mean() / truth.std().clamp(min=1e-6)).item()
    share = blur / ideal if ideal < 0 else 0.0
    light = ((level((images * 1.2).clamp(0, 255)) - base).abs().mean() / spread).item()
    dark = ((level(images * 0.8) - base).abs().mean() / spread).item()
    bright = max(light, dark)
    return {
        "blur_change_vs_spread": round(blur, 3),
        "blur_change_of_measured_detail": round(ideal, 3),
        "blur_share_of_ideal": round(share, 3),
        "brightness_change_vs_spread": round(bright, 3),
        "passes": share >= 0.7 and bright <= 0.2,
    }


def main():
    torch.manual_seed(0)
    if AREA == "skin":
        images_np, valid_np, groups_np = load_scin(40 if QUICK else 150)
        if IMAGES_DIR:
            # Extra skin datasets; their tone group is measured from the photo below.
            more = load_folder(IMAGES_DIR, 200 if QUICK else 8000)
            images_np = np.concatenate([images_np, more[0]])
            valid_np = np.concatenate([valid_np, more[1]])
            groups_np = np.concatenate([groups_np, np.full(len(more[0]), -1)])
    else:
        if not IMAGES_DIR:
            raise SystemExit("Hair and nails need --images DIR (see research/vision/README.md)")
        images_np, valid_np, groups_np = load_folder(IMAGES_DIR, 200 if QUICK else 8000)
        groups_np = np.full(len(images_np), -1)
        if AREA == "hair":
            valid_np = hair_masks(images_np, valid_np)
    order = np.random.default_rng(0).permutation(len(images_np))
    split = int(0.8 * len(order))
    train_ids, test_ids = order[:split], order[split:]
    images = torch.tensor(images_np, dtype=torch.float32).permute(0, 3, 1, 2).to(DEVICE)
    valid = torch.tensor(valid_np, dtype=torch.float32).to(DEVICE)
    groups = torch.tensor(groups_np).to(DEVICE)
    fair = AREA in ("skin", "nails")
    if fair:
        # SCIN photos keep their human-graded Monk group; every other photo is measured (ITA).
        measured = tone_groups(images, valid)
        groups = torch.where(groups < 0, measured, groups)
    else:
        groups = torch.zeros_like(groups)
    counts = [int((groups == k).sum()) for k in range(4)]
    print("photos per skin-tone band:", dict(zip(TONE_NAMES, counts)) if fair else "n/a (hair)")
    model = UNet(len(HEADS)).to(DEVICE)
    print(f"{AREA}: {len(images_np)} photos, heads {HEADS}, {sum(p.numel() for p in model.parameters())} parameters, {DEVICE}")
    optimiser = torch.optim.Adam(model.parameters(), lr=2e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, EPOCHS * STEPS)
    gen = torch.Generator(device=DEVICE).manual_seed(1)
    labelled = None
    if LABELS_DIR:
        if "texture" not in HEADS:
            raise SystemExit("--labels trains the texture head; add texture to --heads")
        patches_np, label_train, label_test, consistency = load_labels(LABELS_DIR)
        labelled = torch.tensor(patches_np, dtype=torch.float32).permute(0, 3, 1, 2).to(DEVICE)
    train_t = torch.tensor(train_ids, device=DEVICE)
    scale = 10.0
    for epoch in range(EPOCHS):
        total = 0.0
        for _ in range(STEPS):
            if fair:
                # Equal photos from every skin-tone band in every batch, however uneven the data.
                parts = []
                for k in range(4):
                    pool = train_t[groups[train_t] == k]
                    if len(pool) > 0:
                        parts.append(pool[torch.randint(len(pool), (10,), device=DEVICE, generator=gen)])
                idx = torch.cat(parts)
            else:
                idx = train_t[torch.randint(len(train_t), (40,), device=DEVICE, generator=gen)]
            xa, xb, every, mask, base = make_batch(images[idx], valid[idx], gen)
            targets = every[:, OWN]
            out_a = model(xa)
            diff = model(xb) - out_a
            per_head = (((diff - targets * scale) ** 2) * mask[:, None]).flatten(2).sum(2).div(mask.flatten(1).sum(1).clamp(min=1)[:, None]).mean(0)
            if BALANCE:
                energy = (((targets * scale) ** 2) * mask[:, None]).flatten(2).sum(2).div(mask.flatten(1).sum(1).clamp(min=1)[:, None]).mean(0)
                per_head = per_head / energy.detach().clamp(min=1e-3)
            loss = per_head.mean()
            if PATCHES and "texture" in HEADS:
                # Readings on one photo must mean something too, not only changes: the texture
                # head's level on view A tracks that photo's own fine contrast (anchor), and every
                # clear change must at least point the right way (ranking).
                k = HEADS.index("texture")
                own = hp_contrast(base, mask) * scale
                level = masked_mean(out_a, mask)[:, k]
                loss = loss + 0.5 * ((level - own) ** 2).mean() / (own**2).mean().clamp(min=1e-3)
                # The same rule on a blurred copy of the same photo, with its own measured detail:
                # the head sees directly that one photo, softened, reads smoother.
                soft = box(base, 3)
                soft_own = hp_contrast(soft, mask) * scale
                soft_level = masked_mean(model(network_input(soft, (valid[idx] > 0).float())), mask)[:, k]
                loss = loss + 0.5 * ((soft_level - soft_own) ** 2).mean() / (own**2).mean().clamp(min=1e-3)
                change = masked_mean(diff, mask)[:, k]
                truth = masked_mean(targets, mask)[:, k] * scale
                clear = truth.abs() > 0.05
                if clear.any():
                    loss = loss + 0.5 * F.relu(0.1 - change[clear] * torch.sign(truth[clear])).mean()
            if fair:
                sample = train_t[torch.randint(len(train_t), (64,), device=DEVICE, generator=gen)]
                m = area_mask(images[sample], valid[sample])
                level = masked_mean(model(network_input(images[sample], (valid[sample] > 0).float())), m)
                g = groups[sample]
                means = torch.stack([level[g == k].mean(0) for k in range(4) if (g == k).any()])
                loss = loss + 2.0 * ((means - means.mean(0)) ** 2).mean()
            if labelled is not None and label_train:
                loss = loss + label_loss(model, labelled, label_train, gen)
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            scheduler.step()
            total += loss.item()
        print(f"epoch {epoch + 1}/{EPOCHS} loss {total / STEPS:.4f}", flush=True)

    model.eval()
    report = {"area": AREA, "name": NAME, "photos": len(images_np), "photos_per_tone_band": counts if fair else None, "heads": {}}
    with torch.no_grad():
        egen = torch.Generator(device=DEVICE).manual_seed(99)
        reps = torch.tensor(test_ids, device=DEVICE).repeat_interleave(8)
        diffs, truths, bands = [], [], []
        for start in range(0, len(reps), 64):
            idx = reps[start : start + 64]
            xa, xb, every, mask, _ = make_batch(images[idx], valid[idx], egen)
            diffs.append(masked_mean(model(xb) - model(xa), mask))
            truths.append(masked_mean(every, mask))
            bands.append(groups[idx])
        diff, truth = torch.cat(diffs).cpu().numpy(), torch.cat(truths).cpu().numpy()
        band = torch.cat(bands).cpu().numpy()
        for k, head in enumerate(HEADS):
            j = ALL_HEADS.index(head)
            d, t = diff[:, k], truth[:, j]
            slope = float(np.polyfit(t, d, 1)[0]) if t.std() > 0 else float("nan")
            # How much this head reacts to the other effects and to light alone (lower is better).
            # Leak is measured against every other effect, including those this model does not read.
            others = np.abs(np.delete(truth, j, axis=1)).sum(1) > 0
            alone = (t == 0)
            report["heads"][head] = {
                "corr_with_true_amount": round(float(np.corrcoef(d, t)[0, 1]), 3),
                "noise_when_absent_in_own_units": round(float(np.std(d[alone]) / abs(slope)), 4) if alone.any() else None,
                # Same check per skin-tone band: a fair model has similar numbers in every band.
                "corr_by_tone_band": [
                    round(float(np.corrcoef(d[band == g], t[band == g])[0, 1]), 3) if (band == g).sum() > 5 and t[band == g].std() > 0 else None
                    for g in range(4)
                ] if fair else None,
                "leak_from_other_effects": round(float(np.corrcoef(d[alone & others], np.delete(truth, j, axis=1)[alone & others].sum(1))[0, 1]), 3) if (alone & others).sum() > 2 else None,
            }
        if PATCHES and "texture" in HEADS:
            report["real_photo_checks"] = real_photo_checks(model, images[torch.tensor(test_ids, device=DEVICE)], valid[torch.tensor(test_ids, device=DEVICE)])
        if labelled is not None:
            agreement = label_agreement(model, labelled, label_test)
            report["labeller"] = {
                "held_out_agreement": None if agreement is None else round(agreement, 3),
                "labeller_self_consistency": None if consistency is None else round(consistency, 3),
                "held_out_pairs": len(label_test),
                # The labeller agreed with herself on 79% of repeated pairs (first real set), so
                # 70% is the realistic bar; a model cannot beat the person it learns from by much.
                "passes": agreement is not None and agreement >= 0.7,
            }
    print(json.dumps(report, indent=2))
    # Weights first: they are all the app needs (the ONNX file can be rebuilt from them), so a
    # failed export or a runtime reset afterwards cannot lose the training.
    weights = {name: t.detach().cpu().numpy().round(6).tolist() for name, t in model.state_dict().items()}
    json.dump({"area": AREA, "name": NAME, "heads": HEADS, "size": SIZE, "detail": DETAIL_GAIN, "weights": weights, "evaluation": report}, open(f"{NAME}_weights.json", "w"))
    json.dump(report, open(f"{NAME}_eval.json", "w"), indent=2)
    print(f"Wrote {NAME}_weights.json and {NAME}_eval.json", flush=True)
    try:
        torch.onnx.export(model.cpu(), torch.zeros(1, IN_CHANNELS, SIZE, SIZE), f"{NAME}_model.onnx", input_names=["input"], output_names=["maps"], opset_version=17, dynamo=False)
        print(f"Wrote {NAME}_model.onnx")
    except Exception as error:  # the weights file is enough
        print(f"ONNX export skipped ({error}); {NAME}_weights.json is all Glowtrace needs")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile fetch_datasets.py
"""Download every Glowtrace training dataset from Roboflow Universe into data/<area>/.

Needs a free Roboflow API key in the ROBOFLOW_API_KEY environment variable (in Colab: the key
icon, "Secrets"). For each dataset it reads the project's details, skips anything whose licence
says non-commercial, downloads the newest version's images, and records name, link and licence
in data/credits.json (CC BY requires credit). Failures are reported and skipped, never fatal.

Usage: python3 fetch_datasets.py [skin|nails|hair ...]
"""
import json
import os
import shutil
import sys
import urllib.request

DATASETS = {
    "skin": [
        ("workspace-1-myjxi", "acne-detection-cbm6h", "Acne, dark circles, pores, wrinkles"),
        ("facial-zfrfm", "facial-ngz7q", "13 facial concerns"),
        ("work-acne", "pigmentation-jiao4", "Pigmentation and spots"),
        ("testworkspace-p2wq3", "face-goyue", "Melasma, marks, pores"),
        ("skinconcerns", "face-wrinkles-detection-xjlkr", "Wrinkles"),
        ("james-aiart", "acne-jcvgb", "Acne"),
        ("acne-severity", "clinic-data", "Acne severity"),
        # Real selfies with marked pores, roughness, dryness, redness and oiliness (CC BY 4.0 on
        # their pages; the licence check below still runs).
        ("faceinc", "dryness-segm-good-selfies", "Selfies: pores, roughness, dryness, redness"),
        ("faceinc", "dryness-segm-preprocessed-good-selfies", "Selfies, preprocessed"),
        ("faceinc", "skin-type-classification", "Faces by skin type"),
        ("frekkel", "facial-skin-analysis", "Facial skin analysis"),
        ("virli", "face-skin", "Face skin"),
    ],
    "nails": [
        ("sergei-artiukhin-ohqpa", "nails-segmentation-yw1zl", "Nail segmentation"),
        ("vincent-y20l1", "fingernail-nqkfm", "Painted and bare nails"),
        ("nailproject", "healthy_nail", "Healthy nails"),
        ("detect-nmkdb", "nail-xaf95", "Nail segmentation"),
        ("aiit-nail-competition", "nail-segmentation-odcgv", "Nail segmentation"),
        ("nsu-sxhmy", "nails-semantic-segmentation", "Nail segmentation"),
        ("personal-projects-jfbag", "nails_segmentation", "Nail segmentation"),
        ("nailproject-padk9", "nail_segmentation", "Nail segmentation"),
        ("fatima-vahabi", "nail-seg-vfa2h", "Nail segmentation"),
        ("vnrvjiet-mnvm9", "nail-images", "Nail conditions"),
        ("nail-acf3b", "nail_disease_own_dataset", "Nail conditions"),
        ("fatima-wmkf6", "nail-segmentation-jrprm", "Nail segmentation"),
    ],
    "hair": [
        ("hairbaldsemanticsegmentation", "hair_baldness", "Scalp and crown"),
        ("umm-alqura-xm8lk", "hairtype", "Curl types incl. 3b and 4A"),
        ("karthik-k", "face-r74ti", "Afro hair, dreadlocks and other styles"),
        ("custom-yolov5-zxmeo", "hair-type", "Straight, wavy, curly"),
        ("flickr", "hair-54plg", "Straight, curly, bald"),
        ("project-kl-wirwir-gg0a3", "hair-style-kpiqp", "Curly, straight, wavy"),
        ("abc-rjytv", "hair_data", "Hair strands"),
        ("itclab", "hair-detection-lkule", "Hair detection"),
        ("hair-oprdr", "mask_segmentation1", "Hair masks"),
        ("manish-kumar-ytl7z", "hair_dataset_updated", "Hair segmentation"),
        # Scalp and hair-health close-ups (found by scout_datasets.ipynb, CC BY 4.0).
        ("sreenithi", "hairproblems-detection", "Good hair, dry scalp, dandruff, oiliness, white hair"),
        ("husnas-workspace", "head-lice-ai", "Healthy scalp and dandruff"),
        ("scalpdet", "scalp-data-iqjst", "Scalp dandruff and oiliness"),
        ("scalpdet", "dandruff-big-chunks", "Dandruff close-ups"),
        ("rooted", "hair-loss-nq8hh-tqveb", "Hair thinning levels"),
        ("major-project-s3rff", "hair-disease-detection-o2ok0", "Scalp conditions"),
    ],
}

# GLOWTRACE_UNSPLASH_ZIP and GLOWTRACE_FETCH_LIMIT exist only for rehearsing the notebook offline.
FETCH_LIMIT = int(os.environ.get("GLOWTRACE_FETCH_LIMIT", "0")) or None

def fetch_bytes(url, timeout=600, tries=4):
    """GET with retries: busy servers (HTTP 429 or 5xx) and dropped connections get three more
    tries after 30 s, 60 s and 120 s, honouring a Retry-After header when the server sends one."""
    import time
    import urllib.error

    for attempt in range(tries):
        try:
            return urllib.request.urlopen(url, timeout=timeout).read()
        except urllib.error.HTTPError as error:
            if error.code not in (429, 500, 502, 503, 504) or attempt == tries - 1:
                raise
            wait = int(error.headers.get("Retry-After") or 0) or 30 * 2**attempt
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == tries - 1:
                raise
            wait = 30 * 2**attempt
        print(f"     busy, retrying in {min(wait, 300)} s: {url[:80]}", flush=True)
        time.sleep(min(wait, 300))
    raise RuntimeError("unreachable")


NON_COMMERCIAL = ("non-commercial", "noncommercial", "-nc", " nc", "by-nc", "private", "research only")


def project_info(workspace, project, key):
    url = f"https://api.roboflow.com/{workspace}/{project}?api_key={key}"
    with urllib.request.urlopen(url, timeout=60) as response:
        return json.loads(response.read().decode())


def licence_of(info):
    """Roboflow does not document a licence field; look in the usual places."""
    project = info.get("project", info)
    for key in ("license", "licence", "license_name"):
        value = project.get(key) or info.get(key)
        if value:
            return str(value)
    return None


ZENODO_SCALP = "https://zenodo.org/api/records/23433121"


def zenodo_scalp(credits):
    """ScalpHairLabels v1 (CC BY 4.0): 504 scalp close-up tiles cut from CC0 ISIC images."""
    import io
    import urllib.request
    import zipfile

    target = os.path.join("data", "hair", "zenodo__scalphairlabels")
    link = "https://zenodo.org/records/23433121"
    try:
        record = json.loads(fetch_bytes(ZENODO_SCALP, timeout=60))
        os.makedirs(target, exist_ok=True)
        for entry in record.get("files", []):
            if "tiles" not in entry["key"]:
                continue
            data = fetch_bytes(entry["links"]["self"])
            zipfile.ZipFile(io.BytesIO(data)).extractall(target)
        count = sum(1 for _, _, files in os.walk(target) for f in files if f.lower().endswith((".jpg", ".jpeg", ".png")))
        print(f"OK   hair: ScalpHairLabels v1 ({count} tiles, licence: CC BY 4.0, source photos CC0)")
        credits = [c for c in credits if c["link"] != link]
        credits.append({"area": "hair", "name": "ScalpHairLabels v1", "what": "Scalp close-ups (ISIC, CC0)",
                        "link": link, "licence": "CC BY 4.0", "images": count})
    except Exception as error:
        print(f"FAIL hair: ScalpHairLabels ({str(error)[:120]})")
    return credits


def fairface(credits, per_group=min(450, FETCH_LIMIT or 450)):
    """FairFace (CC BY 4.0): faces balanced across seven ethnic groups, many real lighting
    conditions. Tight crops (0.25 padding), up to per_group faces per group."""
    import io

    import pandas as pd

    target = os.path.join("data", "skin", "fairface")
    link = "https://github.com/joojs/fairface"
    try:
        files = json.loads(fetch_bytes("https://huggingface.co/api/datasets/HuggingFaceM4/FairFace/parquet", timeout=60))["0.25"]["train"]
        os.makedirs(target, exist_ok=True)
        counts, saved = {}, 0
        for url in files:
            frame = pd.read_parquet(io.BytesIO(fetch_bytes(url, timeout=900)))
            for _, row in frame.iterrows():
                group = str(row.get("race", "unknown"))
                if counts.get(group, 0) >= per_group:
                    continue
                image = row["image"]
                data = image.get("bytes") if isinstance(image, dict) else None
                if not data:
                    continue
                open(os.path.join(target, f"{group}_{counts.get(group, 0):04d}.jpg"), "wb").write(data)
                counts[group] = counts.get(group, 0) + 1
                saved += 1
            if counts and min(counts.values()) >= per_group and len(counts) >= 7:
                break
        print(f"OK   skin: FairFace ({saved} faces, {len(counts)} groups, licence: CC BY 4.0)")
        credits = [c for c in credits if c["link"] != link]
        credits.append({"area": "skin", "name": "FairFace", "what": "Faces balanced across seven ethnic groups",
                        "link": link, "licence": "CC BY 4.0", "images": saved})
    except Exception as error:
        print(f"FAIL skin: FairFace ({str(error)[:120]})")
    return credits


FIGSHARE_NAILS = "https://ndownloader.figshare.com/files/49504041"


def figshare_nails(credits):
    """Skin and fingernails for haemoglobin assessment (Yakimov et al., Scientific Data 2024;
    figshare 25867432, CC0): 252 hand photos of bare nails with nail boxes. Each nail is cut out
    with a 15% margin, the way the nail models read nail tiles."""
    import ast
    import csv
    import io
    import zipfile

    from PIL import Image

    target = os.path.join("data", "nails", "figshare_haemoglobin")
    link = "https://doi.org/10.6084/m9.figshare.25867432.v1"
    try:
        archive = zipfile.ZipFile(io.BytesIO(fetch_bytes(FIGSHARE_NAILS)))
        rows = list(csv.DictReader(io.TextIOWrapper(archive.open("metadata.csv"), "utf-8")))
        os.makedirs(target, exist_ok=True)
        saved = 0
        for row in rows:
            name = f"photo/{row['PATIENT_ID']}.jpg"
            if name not in archive.namelist():
                continue
            # Boxes are [top, left, bottom, right] on the photo as stored (checked by eye on a contact
            # sheet: every crop is a nail); don't apply the EXIF rotation.
            photo = Image.open(io.BytesIO(archive.read(name))).convert("RGB")
            for k, box in enumerate(ast.literal_eval(row["NAIL_BOUNDING_BOXES"])):
                y0, y1 = sorted((box[0], box[2]))
                x0, x1 = sorted((box[1], box[3]))
                if x1 - x0 < 8 or y1 - y0 < 8 or x1 > photo.width or y1 > photo.height:
                    continue
                mx, my = int((x1 - x0) * 0.15), int((y1 - y0) * 0.15)
                tile = photo.crop((max(0, x0 - mx), max(0, y0 - my), min(photo.width, x1 + mx), min(photo.height, y1 + my)))
                tile.save(os.path.join(target, f"{row['PATIENT_ID']}_{k}.jpg"), quality=92)
                saved += 1
        print(f"OK   nails: figshare haemoglobin nails ({saved} nail tiles, licence: CC0)")
        credits = [c for c in credits if c["link"] != link]
        credits.append({"area": "nails", "name": "Skin and fingernails (Yakimov et al. 2024)", "what": "Bare nail tiles from hand photos",
                        "link": link, "licence": "CC0 1.0", "images": saved})
    except Exception as error:
        print(f"FAIL nails: figshare haemoglobin nails ({str(error)[:120]})")
    return credits


OPEN_IMAGES_BOXES = "https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv"
OPEN_IMAGES_META = "https://storage.googleapis.com/openimages/2018_04/validation/validation-images-with-rotation.csv"
HUMAN_HAIR = "/m/03q69"


def open_images_hair(credits, limit=min(1500, FETCH_LIMIT or 1500)):
    """Open Images V7 validation photos with a "Human hair" box (annotations CC BY 4.0). Only images
    whose own licence is CC BY 2.0 are kept; the author and source are recorded per image in
    data/hair/openimages/attribution.csv. Open Images has no hair masks, so hair areas come from
    the hair guide during training, as for every other hair photo."""
    import csv
    import io
    from concurrent.futures import ThreadPoolExecutor

    target = os.path.join("data", "hair", "openimages")
    link = "https://storage.googleapis.com/openimages/web/index.html"
    try:
        boxes = fetch_bytes(OPEN_IMAGES_BOXES).decode()
        with_hair = {r["ImageID"] for r in csv.DictReader(io.StringIO(boxes)) if r["LabelName"] == HUMAN_HAIR}
        meta = fetch_bytes(OPEN_IMAGES_META).decode()
        rows = [
            r for r in csv.DictReader(io.StringIO(meta))
            if r["ImageID"] in with_hair and r["License"].rstrip("/").endswith("licenses/by/2.0")
        ]
        rows = sorted(rows, key=lambda r: r["ImageID"])[:limit]
        os.makedirs(target, exist_ok=True)

        def one(row):
            try:
                url = f"https://open-images-dataset.s3.amazonaws.com/validation/{row['ImageID']}.jpg"
                open(os.path.join(target, f"{row['ImageID']}.jpg"), "wb").write(urllib.request.urlopen(url, timeout=60).read())
                return row
            except Exception:
                return None

        with ThreadPoolExecutor(8) as pool:
            kept = [r for r in pool.map(one, rows) if r]
        with open(os.path.join(target, "attribution.csv"), "w", newline="") as handle:
            writer = csv.writer(handle)
            writer.writerow(["image", "author", "source", "licence"])
            for r in kept:
                writer.writerow([f"{r['ImageID']}.jpg", r["Author"], r["OriginalLandingURL"], r["License"]])
        print(f"OK   hair: Open Images hair ({len(kept)} of {len(rows)} photos, licence: CC BY 2.0 images, CC BY 4.0 annotations)")
        credits = [c for c in credits if c["link"] != link]
        credits.append({"area": "hair", "name": "Open Images V7 (Human hair)", "what": "Photos with hair, per-image credits in attribution.csv",
                        "link": link, "licence": "Images CC BY 2.0; annotations CC BY 4.0", "images": len(kept)})
    except Exception as error:
        print(f"FAIL hair: Open Images hair ({str(error)[:120]})")
    return credits


UNSPLASH_LITE = os.environ.get("GLOWTRACE_UNSPLASH_ZIP", "https://unsplash.com/data/lite/latest")
UNSPLASH_WORDS = {"selfie", "face", "headshot", "portrait", "skin", "close up", "closeup"}


def unsplash_faces(credits, limit=min(1200, FETCH_LIMIT or 1200)):
    """Unsplash Lite (Unsplash Dataset Terms: commercial training for internal business use,
    never redistributed): photos tagged selfie, face, portrait, skin or close-up, 1024 px wide.
    Training keeps only skin-rich patches, so non-face photos drop out on their own."""
    import csv
    import io
    import zipfile
    from concurrent.futures import ThreadPoolExecutor

    target = os.path.join("data", "skin", "unsplash_lite")
    link = "https://unsplash.com/data"
    try:
        archive = zipfile.ZipFile(io.BytesIO(fetch_bytes(UNSPLASH_LITE, timeout=1800)))
        csv.field_size_limit(10**8)
        names = archive.namelist()
        wanted = set()
        with archive.open(next(n for n in names if n.startswith("keywords"))) as handle:
            for row in csv.DictReader(io.TextIOWrapper(handle, "utf-8"), delimiter="\t"):
                if row.get("keyword", "").lower() in UNSPLASH_WORDS:
                    wanted.add(row["photo_id"])
        urls = []
        with archive.open(next(n for n in names if n.startswith("photos"))) as handle:
            for row in csv.DictReader(io.TextIOWrapper(handle, "utf-8"), delimiter="\t"):
                if row.get("photo_id") in wanted and row.get("photo_image_url"):
                    urls.append((row["photo_id"], row["photo_image_url"] + "?w=1024&fm=jpg&q=90"))
        urls = sorted(urls)[:limit]
        os.makedirs(target, exist_ok=True)

        def one(item):
            try:
                data = urllib.request.urlopen(item[1], timeout=60).read()
                open(os.path.join(target, f"{item[0]}.jpg"), "wb").write(data)
                return 1
            except Exception:
                return 0

        with ThreadPoolExecutor(8) as pool:
            saved = sum(pool.map(one, urls))
        print(f"OK   skin: Unsplash Lite faces ({saved} of {len(urls)} photos, licence: Unsplash Dataset Terms, Lite)")
        credits = [c for c in credits if c["link"] != link]
        credits.append({"area": "skin", "name": "Unsplash Lite (faces)", "what": "Selfies, portraits and skin close-ups",
                        "link": link, "licence": "Unsplash Dataset Terms (Lite: commercial internal training, no redistribution)", "images": saved})
    except Exception as error:
        print(f"FAIL skin: Unsplash Lite ({str(error)[:120]})")
    return credits


def main():
    key = os.environ.get("ROBOFLOW_API_KEY")
    # A missing or rejected key skips the Roboflow sets only; every other source still downloads.
    rf = None
    if key and not os.environ.get("GLOWTRACE_SKIP_ROBOFLOW"):
        try:
            from roboflow import Roboflow

            rf = Roboflow(api_key=key)
        except Exception as error:
            print(f"FAIL roboflow: key rejected or service down ({str(error)[:120]}); skipping Roboflow sets")
    else:
        print("SKIP roboflow: no ROBOFLOW_API_KEY")
    areas = [a for a in sys.argv[1:] if a in DATASETS] or list(DATASETS)
    os.makedirs("data", exist_ok=True)
    credits_path = os.path.join("data", "credits.json")
    credits = json.load(open(credits_path)) if os.path.exists(credits_path) else []
    for area in areas:
        for workspace, project, what in DATASETS[area] if rf else []:
            link = f"https://universe.roboflow.com/{workspace}/{project}"
            target = os.path.join("data", area, f"{workspace}__{project}")
            try:
                info = project_info(workspace, project, key)
                licence = licence_of(info)
                if licence and any(word in licence.lower() for word in NON_COMMERCIAL):
                    print(f"SKIP {area}: {project} (licence: {licence})")
                    continue
                versions = rf.workspace(workspace).project(project).versions()
                if not versions:
                    print(f"SKIP {area}: {project} (no downloadable version)")
                    continue
                newest = sorted(versions, key=lambda v: int(str(v.version).split("/")[-1]))[-1]
                if os.path.exists(target):
                    shutil.rmtree(target)
                for fmt in ("coco", "coco-segmentation", "yolov8", "folder"):
                    try:
                        newest.download(fmt, location=target, overwrite=True)
                        break
                    except Exception:
                        continue
                count = sum(
                    1
                    for root, _, files in os.walk(target)
                    for f in files
                    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
                )
                print(f"OK   {area}: {project} ({count} images, licence: {licence or 'not reported, check the page'})")
                credits = [c for c in credits if c["link"] != link]
                credits.append({"area": area, "name": project, "what": what, "link": link, "licence": licence, "images": count})
            except Exception as error:
                print(f"FAIL {area}: {project} ({str(error)[:120]})")
    if "hair" in areas:
        credits = zenodo_scalp(credits)
    if "nails" in areas:
        credits = figshare_nails(credits)
    if "hair" in areas:
        credits = open_images_hair(credits)
    if "skin" in areas:
        credits = fairface(credits)
        credits = unsplash_faces(credits)
    json.dump(credits, open(credits_path, "w"), indent=2)
    print(f"\nCredits written to {credits_path}. Check any 'not reported' licence on its page before shipping.")


if __name__ == "__main__":
    main()

In [ ]:
# 4. Download every skin dataset (about 15-25 minutes): Roboflow sets, FairFace, Unsplash Lite faces.
# Each source that fails is skipped with a FAIL line; the run carries on with the rest.
fetch = subprocess.run(['python', 'fetch_datasets.py', 'skin'], capture_output=True, text=True)
print(fetch.stdout[-6000:]); log(fetch.stdout[-6000:])
if fetch.returncode != 0: log('fetch_datasets stderr:\n' + fetch.stderr[-3000:])
for found in glob.glob(f'{ROOT}/input/**/skin/glowtrace_team', recursive=True):
    shutil.copytree(found, 'data/skin/glowtrace_team', dirs_exist_ok=True)
counts = {d: sum(len(f) for _, _, f in os.walk(f'data/skin/{d}')) for d in sorted(os.listdir('data/skin'))} if os.path.isdir('data/skin') else {}
log('skin photos by source: ' + json.dumps(counts))
if sum(counts.values()) < 200:
    raise SystemExit('Too few skin photos downloaded; send Claude a screenshot of this cell.')

In [ ]:
# 5. Practice run (about 2 minutes): the whole pipeline on a few photos. Any code problem stops here, not hours in.
env = dict(os.environ, GLOWTRACE_EPOCHS='1', GLOWTRACE_STEPS='2', GLOWTRACE_WORK=f'{WORK}/smoke')
smoke = subprocess.run(['python', 'multitask.py', 'skin', '--images', 'data/skin', '--heads', 'texture,shine,breakouts',
        '--name', 'smoke', '--size', '128', '--patches'] + ['--quick'] + (['--labels', LABELS] if LABELS else []), env=env, capture_output=True, text=True)
log('practice run:\n' + smoke.stdout[-1500:])
if smoke.returncode != 0 or not os.path.exists('smoke_weights.json'):
    log(smoke.stderr[-3000:])
    raise SystemExit('Practice run failed: send Claude a screenshot of this cell.')
for f in glob.glob('smoke_*'): os.remove(f)
shutil.rmtree(f'{WORK}/smoke', ignore_errors=True)
log('Practice run passed.')

In [ ]:
# 6. Train the v8 skin surface model (about 1.5-2.5 hours). Output streams to the log as it goes.
name = 'skin-surface'
if os.path.exists(f'{OUT}/{name}_weights.json'):
    log(f'{name}: already done')
else:
    log(f'=== {name} ===')
    process = subprocess.Popen(['python', 'multitask.py', 'skin', '--images', 'data/skin', '--heads', 'texture,shine,breakouts',
        '--name', 'skin-surface', '--size', '128', '--patches'] + (['--labels', LABELS] if LABELS else []), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end=''); LOG.write(line)
    process.wait(); LOG.flush()
    for f in glob.glob(f'{name}_*'): shutil.copy(f, OUT)
    log(f'{name}: exit code {process.returncode}, saved ' + ', '.join(sorted(os.path.basename(f) for f in glob.glob(f'{OUT}/{name}_*'))))

In [ ]:
# 7. One zip with the model, its evaluation, the dataset credits and the full log. Always made.
import zipfile
if os.path.exists('data/credits.json'): shutil.copy('data/credits.json', OUT)
LOG.close()
with zipfile.ZipFile(f'{WORK}/glowtrace_results_v8.zip', 'w') as zf:
    for f in glob.glob(f'{OUT}/*'):
        zf.write(f, os.path.basename(f))
print(sorted(os.path.basename(f) for f in glob.glob(f'{OUT}/*')))
# Keep the output small: Kaggle saves everything in the working folder.
shutil.rmtree('data', ignore_errors=True); shutil.rmtree('cache', ignore_errors=True)
print('Done: glowtrace_results_v8.zip')